# 14 · Export the classical models for the Streamlit demo

Produces three small, verified model files for `models/` in the app repo:

| file | model | features |
|---|---|---|
| `logreg_global6.joblib` | Logistic regression (StandardScaler + LogReg) | the 6 global numbers |
| `rf_global6_demo.joblib` | Random Forest, **demo copy** | the same 6 numbers |
| `rf160_demo.joblib` | Random Forest, **demo copy** | 160 classical features |

plus `classical_spec.json` (feature configuration + each model's own validation macro-F1)
and the MD5 lines for `models/MD5SUMS.txt`.

**Why "demo copy"?** The Step 3.1 forests (300 fully-grown trees on 38,013 rows) weigh several hundred MB
each - too large for GitHub / Streamlit Cloud. This notebook refits the *identical recipe* on the *identical
split and features* with fewer, size-capped trees (`RF_N_TREES`, `RF_MAX_LEAF_NODES`), and records the
resulting validation macro-F1 next to the report's figure so the app can show its own number honestly.
The logistic regression is the exact Step 3.1 recipe (same per-class sample, same seed) - it is refit,
not approximated.

Conventions as in every Deliverable 2 notebook: one config block, `SPLIT_ID` re-derived and asserted first,
`SMOKE_TEST` flag, artefacts saved before anything else is done with them. Runtime on the Colab CPU: ~12 min
(feature extraction dominates); GPU not needed.

In [1]:
# Colab setup: Kaggle credentials from Secrets (with retry) + mount Drive. Harmless when run locally.
import os, time

ON_COLAB = False
try:
    from google.colab import userdata, drive
    ON_COLAB = True
except ModuleNotFoundError:
    print("Not on Colab - using local ~/.kaggle/kaggle.json and a local folder in place of Drive.")

if ON_COLAB:
    ok = False
    for attempt in range(1, 4):
        try:
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
            print(f"Kaggle credentials loaded from Colab Secrets (attempt {attempt}).")
            ok = True
            break
        except Exception as e:
            print(f"  attempt {attempt}: Secrets not ready ({type(e).__name__}); retrying in 3s...")
            time.sleep(3)
    if not ok:
        print("Colab Secrets did not respond - re-run this cell (usually transient).")
    drive.mount("/content/drive")

Kaggle credentials loaded from Colab Secrets (attempt 1).
Mounted at /content/drive


## 1. Config

In [2]:
# ---- the one config block: everything below reads from here ----
from pathlib import Path
from collections import deque
import sys, subprocess, hashlib, random, json
import numpy as np, pandas as pd

SEED               = 42
SPLIT_ID_EXPECTED  = "9e33ec57c1ec"        # canonical split fingerprint - STOP on mismatch
N_CLASSES_EXP      = 38
SIZES_EXP          = (38_013, 8_146)       # train, val rows of the frozen split

# ---- feature configuration: IDENTICAL to Step3_1_PlantVillage_RandomForest ----
FEAT_SIZE    = 64
RGB_BINS     = 32
HSV_BINS     = 16
GLCM_LEVELS  = 32
GLCM_DIST    = [1, 2]
GLCM_ANGLES  = [0, np.pi/4, np.pi/2, 3*np.pi/4]
GLCM_PROPS   = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"]
TAB_COLS     = ["file_size_kb", "brightness", "mean_R", "mean_G", "mean_B", "green_frac"]

# ---- model recipes ----
LOGREG_TRAIN_PER_CLASS = 300     # Step 3.1: LogReg fit on 300 images per class, scored on full val
RF_N_TREES             = 100     # demo forests: fewer trees ...
RF_MAX_LEAF_NODES      = 1000    # ... and size-capped trees (Step 3.1 used 300 fully-grown trees)
MAX_FILE_MB            = 45      # hard ceiling per exported file (GitHub refuses > 100 MB)

# reference figures from Deliverable 2, Table C.1 (validation macro-F1)
REPORT_VAL_F1 = {"logreg": 0.274, "rf6": 0.382, "rf160": 0.906}

# Smoke test: True = 60 images/class, artefacts get a _SMOKE suffix, nothing real is overwritten
SMOKE_TEST      = False
SMOKE_PER_CLASS = 60
_sfx = "_SMOKE" if SMOKE_TEST else ""

random.seed(SEED); np.random.seed(SEED)

# ---- Drive layout ----
_drive     = Path("/content/drive/MyDrive")
DRIVE_ROOT = _drive if _drive.exists() else Path.home()
D2         = DRIVE_ROOT / "plant_recognition" / "deliverable2"
SPLIT_DIR  = D2 / "00_split"
SPLIT_CSV  = {s: SPLIT_DIR / f"split_{s}.csv" for s in ("train", "val", "test")}
OUT_DIR    = D2 / "06_export" / "classical"; OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR  = OUT_DIR / "cache";              CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---- raw images (local disk) ----
PV_DIR   = Path.home() / "plant_recognition" / "plantvillage"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff"}

print("split     :", SPLIT_DIR)
print("output    :", OUT_DIR)
print(f"config    : trees={RF_N_TREES}, max_leaf_nodes={RF_MAX_LEAF_NODES}, "
      f"logreg/class={LOGREG_TRAIN_PER_CLASS}, SMOKE_TEST={SMOKE_TEST}")

split     : /content/drive/MyDrive/plant_recognition/deliverable2/00_split
output    : /content/drive/MyDrive/plant_recognition/deliverable2/06_export/classical
config    : trees=100, max_leaf_nodes=1000, logreg/class=300, SMOKE_TEST=False


## 2. Dataset + the canonical split (fingerprint asserted)

In [3]:
# Download PlantVillage (skips if already on disk), then locate the 'color' rendering
if not PV_DIR.exists() or not any(PV_DIR.iterdir()):
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        from kaggle.api.kaggle_api_extended import KaggleApi
    PV_DIR.mkdir(parents=True, exist_ok=True)
    api = KaggleApi(); api.authenticate()
    api.dataset_download_files("abdallahalidev/plantvillage-dataset",
                               path=str(PV_DIR), unzip=True, quiet=False)
    print("Downloaded to", PV_DIR)
else:
    print("Already present:", PV_DIR)

def locate(root, names, max_depth=4):            # same BFS helper as Step 1 / Step 3.1
    q = deque([(root, 0)])
    while q:
        d, depth = q.popleft()
        if d.is_dir() and d.name.lower() in names:
            return d
        if d.is_dir() and depth < max_depth:
            for c in sorted(d.iterdir()):
                if c.is_dir():
                    q.append((c, depth + 1))
    return None

COLOR_DIR = locate(PV_DIR, {"color"})
assert COLOR_DIR is not None, "Could not find a 'color' folder under PV_DIR - check the download."
class_names = sorted([d.name for d in COLOR_DIR.iterdir() if d.is_dir()])
assert len(class_names) == N_CLASSES_EXP
name2idx = {c: i for i, c in enumerate(class_names)}
print("color dir:", COLOR_DIR, "| classes:", len(class_names))

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset


100%|██████████| 2.04G/2.04G [00:17<00:00, 124MB/s] 



Downloaded to /root/plant_recognition/plantvillage
color dir: /root/plant_recognition/plantvillage/plantvillage dataset/color | classes: 38


In [4]:
# Load the frozen split, re-derive SPLIT_ID, STOP on mismatch
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

assert all(p.exists() for p in SPLIT_CSV.values()), f"split CSVs missing in {SPLIT_DIR}"
digests  = {s: md5(SPLIT_CSV[s]) for s in ("train", "val", "test")}
SPLIT_ID = hashlib.md5("".join(digests[s] for s in ("train", "val", "test")).encode()).hexdigest()[:12]
print("SPLIT_ID =", SPLIT_ID)
assert SPLIT_ID == SPLIT_ID_EXPECTED, f"SPLIT_ID {SPLIT_ID} != {SPLIT_ID_EXPECTED} - NOT the canonical split. STOP."

dfs = {s: pd.read_csv(SPLIT_CSV[s]) for s in ("train", "val")}   # test stays sealed - never read here

# paths were written on an earlier runtime; remap to this machine if needed
def remap(p):
    p = Path(p)
    return p if p.exists() else COLOR_DIR / p.parent.name / p.name
for d in dfs.values():
    d["filepath"] = d["filepath"].map(lambda p: str(remap(p)))
    assert all(Path(p).exists() for p in d["filepath"].head(50)), "image paths do not resolve"

sizes = (len(dfs["train"]), len(dfs["val"]))
assert sizes == SIZES_EXP, f"split sizes {sizes} != {SIZES_EXP}"
p_tr, y_tr   = dfs["train"]["filepath"].values, dfs["train"]["label"].values
p_val, y_val = dfs["val"]["filepath"].values,   dfs["val"]["label"].values

def cap_per_class(paths, labels, cap, seed=SEED):    # identical to Step 3.1
    rng = random.Random(seed); by = {}
    for p, l in zip(paths, labels): by.setdefault(l, []).append(p)
    op, ol = [], []
    for l, ps in by.items():
        sel = ps if len(ps) <= cap else rng.sample(ps, cap)
        op += sel; ol += [l] * len(sel)
    idx = np.arange(len(op)); rng.shuffle(idx)
    op, ol = np.array(op), np.array(ol)
    return op[idx], ol[idx]

if SMOKE_TEST:
    p_tr, y_tr   = cap_per_class(p_tr, y_tr, SMOKE_PER_CLASS)
    p_val, y_val = cap_per_class(p_val, y_val, max(20, SMOKE_PER_CLASS // 3))
    print(f"[SMOKE_TEST] train={len(p_tr):,}, val={len(p_val):,}")
print(f"train / val : {len(p_tr):,} / {len(p_val):,}   (test split not touched)")

SPLIT_ID = 9e33ec57c1ec
train / val : 38,013 / 8,146   (test split not touched)


## 3. Feature extraction - the Step 3.1 code, unchanged

`extract_global6` is the six-number summary from Deliverable 1 (full resolution, true file size).
`extract_feat160` is the Random-Forest descriptor (histograms + GLCM on a 64×64 copy, plus the six).
Both are cached on Drive so a re-run is instant. The app's `streamlit/features.py` is a line-for-line
port of these two functions.

In [5]:
from PIL import Image
try:
    from skimage.color import rgb2hsv, rgb2gray
    from skimage.feature import graycomatrix, graycoprops
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
    from skimage.color import rgb2hsv, rgb2gray
    from skimage.feature import graycomatrix, graycoprops

def norm_hist(values, bins, rng):
    h = np.histogram(values, bins=bins, range=rng)[0].astype("float64")
    s = h.sum()
    return h / s if s > 0 else h

def extract_global6(fp):
    im = np.asarray(Image.open(fp).convert("RGB")).astype("float64")
    r, g, b = (im[:, :, i].mean() for i in range(3)); br = im[:, :, :3].mean()
    return np.array([os.path.getsize(fp) / 1024, br, r, g, b, g / (r + g + b)])

def extract_feat160(fp):
    a0 = np.asarray(Image.open(fp).convert("RGB")).astype("float64")
    r, g, b = (a0[:, :, i].mean() for i in range(3)); br = a0[:, :, :3].mean()
    global6 = np.array([os.path.getsize(fp) / 1024, br, r, g, b, g / (r + g + b)])
    arr = np.asarray(Image.fromarray(a0.astype("uint8")).resize((FEAT_SIZE, FEAT_SIZE))).astype("float64")
    rgb_hist = np.concatenate([norm_hist(arr[:, :, c], RGB_BINS, (0, 255)) for c in range(3)])
    hsv = rgb2hsv(arr / 255.0)
    hsv_hist = np.concatenate([norm_hist(hsv[:, :, c], HSV_BINS, (0, 1)) for c in range(3)])
    gray = (rgb2gray(arr / 255.0) * (GLCM_LEVELS - 1)).astype("uint8")
    glcm = graycomatrix(gray, distances=GLCM_DIST, angles=GLCM_ANGLES,
                        levels=GLCM_LEVELS, symmetric=True, normed=True)
    prop = {p: graycoprops(glcm, p).mean(axis=1) for p in GLCM_PROPS}
    glcm_feats = np.array([prop[p][di] for di in range(len(GLCM_DIST)) for p in GLCM_PROPS])
    return np.concatenate([rgb_hist, hsv_hist, glcm_feats, global6]).astype("float32")

N_FEAT160 = 3 * RGB_BINS + 3 * HSV_BINS + len(GLCM_DIST) * len(GLCM_PROPS) + 6
assert extract_feat160(p_tr[0]).shape == (N_FEAT160,) and N_FEAT160 == 160
assert np.allclose(extract_feat160(p_tr[0])[-6:], extract_global6(p_tr[0]), rtol=1e-5)
print("feature functions OK: feat160 =", N_FEAT160, "| global6 =", len(TAB_COLS))

def featurize(paths, labels, fn, n_feat, cache, tag):
    """Run fn over paths -> (X, y_int); cached to Drive unless SMOKE_TEST."""
    if not SMOKE_TEST and cache.exists():
        d = np.load(cache)
        print(f"  {tag}: loaded cache {d['X'].shape} <- {cache.name}")
        return d["X"], d["y"]
    X = np.zeros((len(paths), n_feat), dtype="float64"); ok = np.ones(len(paths), bool)
    t0 = time.time()
    for i, fp in enumerate(paths):
        try:
            X[i] = fn(fp)
        except Exception:
            ok[i] = False
        if (i + 1) % 5000 == 0:
            print(f"    {tag}: {i+1:,}/{len(paths):,}  ({time.time()-t0:.0f}s)")
    y = np.array([name2idx[l] for l in labels], dtype="int64")
    X, y = X[ok], y[ok]
    if not SMOKE_TEST:
        np.savez_compressed(cache, X=X, y=y)
    print(f"  {tag}: {X.shape} in {time.time()-t0:.0f}s")
    return X, y

print("\nextracting (one-off, cached on Drive)...")
X6_tr,   y6_tr   = featurize(p_tr,  y_tr,  extract_global6, 6,   CACHE_DIR / "global6_train.npz", "global6 train")
X6_val,  y6_val  = featurize(p_val, y_val, extract_global6, 6,   CACHE_DIR / "global6_val.npz",   "global6 val")
X160_tr, y160_tr = featurize(p_tr,  y_tr,  extract_feat160, 160, CACHE_DIR / "feat160_train.npz", "feat160 train")
X160_val,y160_val= featurize(p_val, y_val, extract_feat160, 160, CACHE_DIR / "feat160_val.npz",   "feat160 val")

feature functions OK: feat160 = 160 | global6 = 6

extracting (one-off, cached on Drive)...
    global6 train: 5,000/38,013  (6s)
    global6 train: 10,000/38,013  (16s)
    global6 train: 15,000/38,013  (23s)
    global6 train: 20,000/38,013  (31s)
    global6 train: 25,000/38,013  (37s)
    global6 train: 30,000/38,013  (45s)
    global6 train: 35,000/38,013  (52s)
  global6 train: (38013, 6) in 57s
    global6 val: 5,000/8,146  (7s)
  global6 val: (8146, 6) in 11s
    feat160 train: 5,000/38,013  (41s)
    feat160 train: 10,000/38,013  (86s)
    feat160 train: 15,000/38,013  (129s)
    feat160 train: 20,000/38,013  (171s)
    feat160 train: 25,000/38,013  (212s)
    feat160 train: 30,000/38,013  (255s)
    feat160 train: 35,000/38,013  (297s)
  feat160 train: (38013, 160) in 326s
    feat160 val: 5,000/8,146  (39s)
  feat160 val: (8146, 160) in 66s


## 4. Fit the three models, score on the full validation set

- **LogReg** - exactly Step 3.1: `cap_per_class(train, 300, seed=42)` → StandardScaler → LogisticRegression(balanced). Deterministic, so it should land on the report's 0.274.
- **RF-6 demo** and **RF-160 demo** - the Step 3.1 recipe (`class_weight="balanced"`, seed 42) with `RF_N_TREES` size-capped trees on the full training split. Their own validation macro-F1 is written to the spec and shown in the app.

The test split is not read anywhere in this notebook.

In [6]:
import sklearn, skimage, joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

def score(model, X, y):
    pred = model.predict(X)
    return round(float(f1_score(y, pred, average="macro", zero_division=0)), 4), round(float(accuracy_score(y, pred)), 4)

metrics = {}

# ---- LogReg on the Step 3.1 per-class train sample ----
ps_lr, ls_lr = cap_per_class(p_tr, y_tr, LOGREG_TRAIN_PER_CLASS)
row_of = {p: i for i, p in enumerate(p_tr)}
sel = np.array([row_of[p] for p in ps_lr])
logreg = Pipeline([("scaler", StandardScaler()),
                   ("logreg", LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1))])
t0 = time.time(); logreg.fit(X6_tr[sel], y6_tr[sel])
metrics["logreg"] = score(logreg, X6_val, y6_val)
print(f"LogReg  (6 feats, {len(sel):,} train rows) : val macro-F1 {metrics['logreg'][0]:.4f} | acc {metrics['logreg'][1]:.4f} "
      f"| report {REPORT_VAL_F1['logreg']} | {time.time()-t0:.0f}s")

# ---- RF-6 demo copy ----
rf6 = RandomForestClassifier(n_estimators=RF_N_TREES, max_leaf_nodes=RF_MAX_LEAF_NODES,
                             class_weight="balanced", n_jobs=-1, random_state=SEED)
t0 = time.time(); rf6.fit(X6_tr, y6_tr)
metrics["rf6"] = score(rf6, X6_val, y6_val)
print(f"RF-6    demo ({RF_N_TREES} trees)           : val macro-F1 {metrics['rf6'][0]:.4f} | acc {metrics['rf6'][1]:.4f} "
      f"| report {REPORT_VAL_F1['rf6']} | {time.time()-t0:.0f}s")

# ---- RF-160 demo copy ----
rf160 = RandomForestClassifier(n_estimators=RF_N_TREES, max_leaf_nodes=RF_MAX_LEAF_NODES,
                               class_weight="balanced", n_jobs=-1, random_state=SEED)
t0 = time.time(); rf160.fit(X160_tr, y160_tr)
metrics["rf160"] = score(rf160, X160_val, y160_val)
print(f"RF-160  demo ({RF_N_TREES} trees)           : val macro-F1 {metrics['rf160'][0]:.4f} | acc {metrics['rf160'][1]:.4f} "
      f"| report {REPORT_VAL_F1['rf160']} | {time.time()-t0:.0f}s")

for k in ("logreg", "rf6", "rf160"):
    assert len(getattr(logreg if k == "logreg" else (rf6 if k == "rf6" else rf160), "classes_")) == 38
if not SMOKE_TEST:
    gap = REPORT_VAL_F1["rf160"] - metrics["rf160"][0]
    print(f"\nRF-160 demo copy vs report: {gap:+.3f}  (expected small; raise RF_MAX_LEAF_NODES if it is > 0.03)")

LogReg  (6 feats, 10,967 train rows) : val macro-F1 0.2739 | acc 0.3285 | report 0.274 | 5s
RF-6    demo (100 trees)           : val macro-F1 0.3796 | acc 0.4567 | report 0.382 | 19s
RF-160  demo (100 trees)           : val macro-F1 0.8592 | acc 0.8844 | report 0.906 | 49s

RF-160 demo copy vs report: +0.047  (expected small; raise RF_MAX_LEAF_NODES if it is > 0.03)


## 5. Save, measure, verify, write the spec

Compressed joblib; each file must stay under `MAX_FILE_MB`. Reloaded and checked against the in-memory
model before the spec is written.

In [7]:
files = {"logreg": OUT_DIR / f"logreg_global6{_sfx}.joblib",
         "rf6":    OUT_DIR / f"rf_global6_demo{_sfx}.joblib",
         "rf160":  OUT_DIR / f"rf160_demo{_sfx}.joblib"}
models = {"logreg": logreg, "rf6": rf6, "rf160": rf160}

for k, m in models.items():
    joblib.dump(m, files[k], compress=("zlib", 3))
    mb = files[k].stat().st_size / 1e6
    print(f"{files[k].name:<28} {mb:6.1f} MB")
    assert mb < MAX_FILE_MB, f"{files[k].name} is {mb:.0f} MB - lower RF_N_TREES / RF_MAX_LEAF_NODES and re-run"

# reload and prove the file predicts exactly like the fitted model
Xchk = {"logreg": X6_val[:200], "rf6": X6_val[:200], "rf160": X160_val[:200]}
for k, f in files.items():
    back = joblib.load(f)
    assert np.array_equal(back.predict(Xchk[k]), models[k].predict(Xchk[k])), f"{k}: reloaded file differs"
print("reload check OK - files predict identically to the fitted models")

spec = {
    "split_id": SPLIT_ID,
    "sklearn": sklearn.__version__,
    "scikit_image": skimage.__version__,
    "global6": {"columns": TAB_COLS,
                "note": "full-resolution image; file_size_kb = true file size in KB"},
    "feat160": {"feat_size": FEAT_SIZE, "rgb_bins": RGB_BINS, "hsv_bins": HSV_BINS,
                "glcm_levels": GLCM_LEVELS, "glcm_dist": GLCM_DIST, "glcm_props": GLCM_PROPS,
                "n_features": 160,
                "note": "histograms + GLCM on a 64x64 copy; the 6 global numbers on the full image"},
    "models": {
        "logreg": {"file": files["logreg"].name, "features": "global6",
                   "recipe": f"Step 3.1 exact: {LOGREG_TRAIN_PER_CLASS}/class train sample, StandardScaler, LogisticRegression(balanced)",
                   "val_macro_f1": metrics["logreg"][0], "val_accuracy": metrics["logreg"][1],
                   "report_val_macro_f1": REPORT_VAL_F1["logreg"]},
        "rf6":    {"file": files["rf6"].name, "features": "global6",
                   "recipe": f"demo copy: {RF_N_TREES} trees, max_leaf_nodes={RF_MAX_LEAF_NODES}, balanced, full train split (report: 300 fully-grown trees)",
                   "val_macro_f1": metrics["rf6"][0], "val_accuracy": metrics["rf6"][1],
                   "report_val_macro_f1": REPORT_VAL_F1["rf6"]},
        "rf160":  {"file": files["rf160"].name, "features": "feat160",
                   "recipe": f"demo copy: {RF_N_TREES} trees, max_leaf_nodes={RF_MAX_LEAF_NODES}, balanced, full train split (report: 300 fully-grown trees)",
                   "val_macro_f1": metrics["rf160"][0], "val_accuracy": metrics["rf160"][1],
                   "report_val_macro_f1": REPORT_VAL_F1["rf160"]},
    },
    "smoke_test": SMOKE_TEST,
}
spec_path = OUT_DIR / f"classical_spec{_sfx}.json"
spec_path.write_text(json.dumps(spec, indent=2))

print("\n==== copy these 4 lines into models/MD5SUMS.txt ====")
for f in list(files.values()) + [spec_path]:
    print(f"{md5(f)}  {f.name}")
print("\n==== copy these 2 lines into requirements.txt ====")
print(f"scikit-learn=={sklearn.__version__}")
print(f"scikit-image=={skimage.__version__}")
print("\nfiles are in Drive:", OUT_DIR)

logreg_global6.joblib           0.0 MB
rf_global6_demo.joblib         22.7 MB
rf160_demo.joblib              20.2 MB
reload check OK - files predict identically to the fitted models

==== copy these 4 lines into models/MD5SUMS.txt ====
d1a15c8ea22d5259249cf168aba99c2f  logreg_global6.joblib
03e437faab934ba33ca06c409dcfe6d4  rf_global6_demo.joblib
e69565ebbbdb12705021913350196a64  rf160_demo.joblib
81db06e99619c2dc054774f76e6125e5  classical_spec.json

==== copy these 2 lines into requirements.txt ====
scikit-learn==1.6.1
scikit-image==0.25.2

files are in Drive: /content/drive/MyDrive/plant_recognition/deliverable2/06_export/classical


## 6. What to do next

1. Download the four files (`logreg_global6.joblib`, `rf_global6_demo.joblib`, `rf160_demo.joblib`,
   `classical_spec.json`) from Drive `deliverable2/06_export/classical/` into the repo folder `models/`.
2. Paste the four MD5 lines into `models/MD5SUMS.txt` and the two version pins into `requirements.txt`.
3. Commit and push; the demo app loads them, MD5-verified, on first visit.

`_SMOKE` files are dry-run artefacts only - never copy them into the repo.